# MNPS Job Classification Likelihood & Severity-Weighted Evaluation (v8.2 Integrated)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

This notebook integrates the strengths of three previous notebooks:

- **v7 Likelihood Scorer – Complete Integration** (batch-oriented likelihood & confidence)
- **v4 KSAC-Aware Likelihood Scorer** (richer KSAC & confidence signals)
- **Eval_SeverityWeighted_v2** (severity-weighted confusion matrices)

It is designed for **real data** and assumes you will provide:

- `Evaluation Resources.zip` containing:
  - `MNPS KSACs.csv`
  - `MNPS_Role_Groups_by_KSAC_Similarity_FINAL.csv`
  - `salary_by_major_role_grouping.csv`
  - `Time to correct an error in hours.csv`
- `Sample_JDs.csv` (job description components)
- `Job_Classifications_Batch.csv` (model classifications, including at minimum a major role group column)



In [ ]:
# 1️⃣ Imports & base paths

import os
import io
import json
import math
import zipfile
import shutil
from datetime import datetime

import numpy as np
import pandas as pd

try:
    from google.colab import drive, files  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    class DummyFiles:
        def upload(self):
            raise RuntimeError("`files.upload()` is only available in Colab. Please run this notebook in Colab or adapt the I/O section.")
    files = DummyFiles()

print(f"Running in Colab: {IN_COLAB}")

BASE_PATH = "/content" if IN_COLAB else os.getcwd()
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")

RUN_FOLDER = os.path.join(BASE_PATH, f\"MNPS_Likelihood_Run_{RUN_TAG}\")
INPUTS_PATH = os.path.join(RUN_FOLDER, "inputs")
RESOURCES_PATH = os.path.join(RUN_FOLDER, "resources")
RESULTS_PATH = os.path.join(RUN_FOLDER, "results")

for p in [RUN_FOLDER, INPUTS_PATH, RESOURCES_PATH, RESULTS_PATH]:
    os.makedirs(p, exist_ok=True)

print("📁 Run folder:", RUN_FOLDER)
print("📁 Inputs path:", INPUTS_PATH)
print("📁 Resources path:", RESOURCES_PATH)
print("📁 Results path:", RESULTS_PATH)


In [ ]:
# 2️⃣ Configuration – thresholds & weights

HUMAN_BASELINE_LOW = 0.88
HUMAN_BASELINE_HIGH = 0.94

LIKELIHOOD_MIN = 0.0
LIKELIHOOD_MAX = 5.0

WEIGHT_KSAC_SIMILARITY = 0.35
WEIGHT_SALARY_IMPACT   = 0.25
WEIGHT_TIME_IMPACT     = 0.15
WEIGHT_CONFIDENCE      = 0.25

CRITICAL_ERROR_THRESHOLD = 0.65
MAJOR_ERROR_THRESHOLD    = 0.45
MINOR_ERROR_THRESHOLD    = 0.20

BASE_CORRECTION_TIME = 8.0
CRITICAL_TIME_MULTIPLIER = 3.0
MAJOR_TIME_MULTIPLIER    = 2.0
MINOR_TIME_MULTIPLIER    = 1.3

SEVERITY_SALARY_WEIGHT = 0.7
SEVERITY_TIME_WEIGHT   = 0.3

TRUE_LABEL_COL  = "true_major_group"
PRED_LABEL_COL  = "major_role_group"

TRUE_SUBGROUP_COL = ""
PRED_SUBGROUP_COL = ""

print("✅ Configuration loaded.")


In [ ]:
# 3️⃣ (Optional) Mount Google Drive for persistent storage

if IN_COLAB:
    try:
        drive.mount('/content/drive')
        print("✅ Google Drive mounted at /content/drive")
    except Exception as e:
        print("⚠️ Could not mount Google Drive:", e)
else:
    print("ℹ️ Not running in Colab; skipping Drive mount.")


In [ ]:
# 4️⃣ Upload files and load batch data

print("📤 Please upload the following files:")
print("1. Evaluation Resources.zip (contains KSACs, salary, time, etc.)")
print("2. Sample_JDs.csv (job descriptions)")
print("3. Job_Classifications_Batch.csv (model classifications)\n")

uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\n📁 Processing: {filename}")
    if filename.lower().endswith(".zip"):
        with zipfile.ZipFile(io.BytesIO(uploaded[filename]), "r") as zip_ref:
            zip_ref.extractall(RESOURCES_PATH)
        print("   ✅ Extracted Evaluation Resources to:", RESOURCES_PATH)
    elif filename.lower().endswith(".csv"):
        with open(os.path.join(INPUTS_PATH, filename), "wb") as f:
            f.write(uploaded[filename])
        print("   ✅ Copied CSV to inputs:", os.path.join(INPUTS_PATH, filename))
    else:
        print("   ⚠️ Unrecognized file type; copying to inputs for manual handling.")
        with open(os.path.join(INPUTS_PATH, filename), "wb") as f:
            f.write(uploaded[filename])

def find_file(patterns, path):
    patterns = [p.lower() for p in patterns]
    for fname in os.listdir(path):
        lower = fname.lower()
        if all(p in lower for p in patterns):
            return os.path.join(path, fname)
    return None

class_path = os.path.join(INPUTS_PATH, "Job_Classifications_Batch.csv")
if not os.path.exists(class_path):
    fallback = find_file(["class"], INPUTS_PATH)
    if fallback:
        print(f"⚠️ Job_Classifications_Batch.csv not found; using {os.path.basename(fallback)} instead.")
        class_path = fallback
    else:
        raise FileNotFoundError("No classification file found. Please upload Job_Classifications_Batch.csv.")

classifications = pd.read_csv(class_path)
print(f"✅ Loaded classifications from {os.path.basename(class_path)} ({len(classifications)} rows)")

jd_path = os.path.join(INPUTS_PATH, "Sample_JDs.csv")
if not os.path.exists(jd_path):
    fallback = find_file(["jd"], INPUTS_PATH)
    if fallback:
        print(f"⚠️ Sample_JDs.csv not found; using {os.path.basename(fallback)} instead.")
        jd_path = fallback
    else:
        jd_path = None
        print("⚠️ No job description CSV found; proceeding with classifications only.")

job_descriptions = pd.read_csv(jd_path) if jd_path else None
if job_descriptions is not None:
    print(f"✅ Loaded job descriptions from {os.path.basename(jd_path)} ({len(job_descriptions)} rows)")
else:
    print("ℹ️ job_descriptions is None")

merged_data = classifications.copy()
if job_descriptions is not None and "source_row_index" in merged_data.columns:
    merged_data = merged_data.merge(
        job_descriptions,
        left_on="source_row_index",
        right_index=True,
        how="left",
        suffixes=("", "_jd")
    )
    print(f"✅ Merged classifications with job descriptions. Final rows: {len(merged_data)}")
else:
    print("ℹ️ Could not merge classifications with job descriptions (missing job_descriptions or source_row_index).")

print("\n✅ Data ready for resource loading and analysis.")


In [ ]:
# 5️⃣ Load Evaluation Resources (KSACs, Role Groups, Salary, Time-to-Correct)

resources = {}

def safe_read_csv(path):
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"⚠️ Failed to read {path}: {e}")
        return None

def find_resource_file(patterns):
    return find_file(patterns, RESOURCES_PATH)

ksac_path = find_resource_file(["ksac"])
if ksac_path:
    ksacs_df = safe_read_csv(ksac_path)
    if ksacs_df is not None:
        resources["ksacs"] = ksacs_df
        print(f"✅ Loaded KSACs from: {os.path.basename(ksac_path)} (rows={len(ksacs_df)})")
else:
    resources["ksacs"] = pd.DataFrame()
    print("⚠️ No KSAC file found in resources. Some KSAC-based features will be synthetic.")

role_groups_path = find_resource_file(["role_groups", "similarity"])
if not role_groups_path:
    role_groups_path = find_resource_file(["role_groups"])
    if not role_groups_path:
        role_groups_path = find_resource_file(["similarity"])
if role_groups_path:
    role_groups_df = safe_read_csv(role_groups_path)
    if role_groups_df is not None:
        resources["role_groups"] = role_groups_df
        print(f"✅ Loaded Role Groups by KSAC Similarity from: {os.path.basename(role_groups_path)}")

salary_path = find_resource_file(["salary_by_major_role_grouping"])
if not salary_path:
    salary_path = find_resource_file(["salary"])
if salary_path:
    salary_df = safe_read_csv(salary_path)
    if salary_df is not None:
        resources["salary_data"] = salary_df
        print(f"✅ Loaded salary data from: {os.path.basename(salary_path)} (rows={len(salary_df)})")
else:
    resources["salary_data"] = pd.DataFrame()
    print("⚠️ No salary file found. Salary-related metrics will use defaults.")

time_path = find_resource_file(["time", "correct"])
if time_path:
    time_df = safe_read_csv(time_path)
    if time_df is not None:
        resources["time_data"] = time_df
        print(f"✅ Loaded time-to-correct data from: {os.path.basename(time_path)} (rows={len(time_df)})")
else:
    print("ℹ️ No explicit time-to-correct file found; using BASE_CORRECTION_TIME defaults.")

print("\n✅ Evaluation resources loaded.")


In [ ]:
# 6️⃣ Helper functions – KSAC similarity, salary & time impact, confidence, composite, likelihood

from math import sqrt

def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1, dtype=float)
    vec2 = np.array(vec2, dtype=float)
    if vec1.shape != vec2.shape or vec1.size == 0:
        return 0.0
    denom = (np.linalg.norm(vec1) * np.linalg.norm(vec2))
    if denom == 0:
        return 0.0
    return float(np.dot(vec1, vec2) / denom)

def calculate_ksac_similarity(role1, role2, ksac_df):
    if ksac_df is None or ksac_df.empty:
        return 0.0
    df = ksac_df.copy()
    df.columns = df.columns.str.strip()

    role_cols = [c for c in df.columns if "role" in c.lower() or "title" in c.lower()]
    if not role_cols:
        return 0.0
    rcol = role_cols[0]

    def get_vector(rname):
        subset = df[df[rcol].astype(str).str.strip() == str(rname).strip()]
        if subset.empty:
            return None
        num_cols = subset.select_dtypes(include=[np.number]).columns
        if len(num_cols) >= 3:
            return subset[num_cols].iloc[0].values
        vals = []
        for c in df.columns:
            if c == rcol:
                continue
            vals.append(1.0 if not pd.isna(subset.iloc[0][c]) and str(subset.iloc[0][c]).strip() != "" else 0.0)
        return np.array(vals, dtype=float)

    v1 = get_vector(role1)
    v2 = get_vector(role2)
    if v1 is None or v2 is None:
        return 0.0
    return cosine_similarity(v1, v2)

def find_alternative_roles(assigned_role, ksac_df, top_n=5):
    if ksac_df is None or ksac_df.empty:
        return []
    df = ksac_df.copy()
    df.columns = df.columns.str.strip()
    role_cols = [c for c in df.columns if "role" in c.lower() or "title" in c.lower()]
    if not role_cols:
        return []
    rcol = role_cols[0]

    roles = df[rcol].dropna().astype(str).unique().tolist()
    target = str(assigned_role)

    sims = []
    for r in roles:
        if r == target:
            continue
        s = calculate_ksac_similarity(target, r, df)
        sims.append((r, s))

    sims.sort(key=lambda x: x[1], reverse=True)
    return sims[:top_n]

def calculate_error_severity(assigned_role, ksac_df):
    alts = find_alternative_roles(assigned_role, ksac_df, top_n=5)
    if not alts:
        return 0.5, [], 0.5
    best_sim = max(s for _, s in alts)
    best_sim = min(max(best_sim, 0.0), 1.0)
    error_severity = 1.0 - best_sim
    return float(error_severity), alts, best_sim

def calculate_salary_amount(major_role_group, salary_df):
    if salary_df is None or salary_df.empty:
        return 60000.0
    df = salary_df.copy()
    df.columns = df.columns.str.strip()

    role_cols = [c for c in df.columns if "major" in c.lower() and "role" in c.lower()]
    if not role_cols:
        role_cols = [df.columns[0]]
    rcol = role_cols[0]

    value_cols = [c for c in df.columns if "salary" in c.lower() or "average" in c.lower() or "annual" in c.lower()]
    if not value_cols:
        value_cols = [df.columns[-1]]
    vcol = value_cols[0]

    subset = df[df[rcol].astype(str).str.strip() == str(major_role_group).strip()]
    if subset.empty:
        return 60000.0

    try:
        raw = subset.iloc[0][vcol]
        if isinstance(raw, str):
            raw = raw.replace("$", "").replace(",", "").strip()
        val = float(raw)
        return val if val > 0 else 60000.0
    except Exception:
        return 60000.0

def estimate_correction_time(error_severity, major_role_group, time_df):
    base_from_table = None
    if time_df is not None and not time_df.empty:
        df = time_df.copy()
        df.columns = df.columns.str.strip()
        role_cols = [c for c in df.columns if "role" in c.lower() or "major" in c.lower()]
        hour_cols = [c for c in df.columns if "hour" in c.lower()]
        if role_cols and hour_cols:
            rcol = role_cols[0]
            hcol = hour_cols[0]
            match = df[df[rcol].astype(str).str.strip() == str(major_role_group).strip()]
            if not match.empty:
                try:
                    base_from_table = float(match.iloc[0][hcol])
                except Exception:
                    base_from_table = None

    base_role_time = base_from_table if base_from_table is not None else BASE_CORRECTION_TIME

    if error_severity >= CRITICAL_ERROR_THRESHOLD:
        base_time = base_role_time * CRITICAL_TIME_MULTIPLIER
    elif error_severity >= MAJOR_ERROR_THRESHOLD:
        base_time = base_role_time * MAJOR_TIME_MULTIPLIER
    elif error_severity >= MINOR_ERROR_THRESHOLD:
        base_time = base_role_time * MINOR_TIME_MULTIPLIER
    else:
        base_time = base_role_time

    complexity = {
        "Director": 2.0, "Manager": 1.8, "Specialist": 1.5,
        "Analyst": 1.4, "Counselor": 1.4, "Coach": 1.3,
        "Teacher": 1.3, "Coordinator": 1.2, "Accountant": 1.2,
        "Technician": 1.1, "Instructor": 1.0, "Assistant": 0.8,
        "Liaison": 1.3
    }.get(str(major_role_group), 1.0)

    return float(base_time * complexity)

def compute_confidence_components(row):
    '''
    Return confidence_score [0–1] and a brief reasoning string.
    Looks at justification length, presence of role keywords, and KSAC alignment proxy.
    '''
    justification_cols = [c for c in row.index if "justification" in c.lower() or "rationale" in c.lower()]
    justification_text = ""
    if justification_cols:
        justification_text = str(row[justification_cols[0]]) if not pd.isna(row[justification_cols[0]]) else ""
    major_role = str(row.get(PRED_LABEL_COL, "")).strip()

    length = len(justification_text.split())
    if length == 0:
        just_score = 0.2
    elif length < 40:
        just_score = 0.5
    elif length < 120:
        just_score = 0.8
    else:
        just_score = 0.9

    lower_just = justification_text.lower()
    indicators = [
        major_role.lower(),
        "scope", "report", "supervis", "team", "strategy", "program", "student", "school"
    ]
    hit = any(tok in lower_just for tok in indicators if tok)

    role_signal_score = 0.8 if hit else 0.4

    error_severity = row.get("error_severity", np.nan)
    if pd.isna(error_severity):
        ksac_align_score = 0.6
    else:
        ksac_align_score = float(1.0 - min(max(error_severity, 0.0), 1.0))

    confidence_score = (
        0.4 * just_score +
        0.3 * role_signal_score +
        0.3 * ksac_align_score
    )
    confidence_score = float(min(max(confidence_score, 0.0), 1.0))

    bands = []
    if length == 0:
        bands.append("no justification text")
    elif length < 40:
        bands.append("very brief justification")
    else:
        bands.append("sufficient justification detail")

    bands.append("clear role signals" if hit else "few explicit role signals")

    if error_severity is not None and not pd.isna(error_severity):
        if error_severity < MINOR_ERROR_THRESHOLD:
            bands.append("strong KSAC alignment proxy")
        elif error_severity < MAJOR_ERROR_THRESHOLD:
            bands.append("moderate KSAC alignment proxy")
        else:
            bands.append("weak KSAC alignment proxy")

    reasoning = "; ".join(bands)
    return confidence_score, reasoning

def compute_composite_quality(ksac_similarity, salary_amount, correction_hours, confidence_score,
                              all_salary_values=None, all_hours_values=None):
    ksac_score = float(min(max(ksac_similarity, 0.0), 1.0))

    if all_salary_values is None or len(all_salary_values) == 0:
        salary_score = 0.7
    else:
        smax = max(all_salary_values)
        sval = min(max(salary_amount, 0.0), smax)
        salary_norm = sval / smax if smax > 0 else 0.0
        salary_score = 1.0 - (salary_norm * ksac_score)

    if all_hours_values is None or len(all_hours_values) == 0:
        time_score = 0.7
    else:
        hmax = max(all_hours_values)
        hval = min(max(correction_hours, 0.0), hmax)
        hours_norm = hval / hmax if hmax > 0 else 0.0
        time_score = 1.0 - hours_norm

    conf_score = float(min(max(confidence_score, 0.0), 1.0))

    composite = (
        WEIGHT_KSAC_SIMILARITY * ksac_score +
        WEIGHT_SALARY_IMPACT   * salary_score +
        WEIGHT_TIME_IMPACT     * time_score +
        WEIGHT_CONFIDENCE      * conf_score
    )
    return float(min(max(composite, 0.0), 1.0))

def map_composite_to_likelihood(composite):
    c = float(min(max(composite, 0.0), 1.0))
    if c <= 0.2:
        return 0.0 + (c / 0.2) * 1.0
    if c <= HUMAN_BASELINE_LOW:
        return 1.0 + ((c - 0.2) / (HUMAN_BASELINE_LOW - 0.2)) * 3.0
    if c <= HUMAN_BASELINE_HIGH:
        return 4.0 + ((c - HUMAN_BASELINE_LOW) / (HUMAN_BASELINE_HIGH - HUMAN_BASELINE_LOW)) * 0.75
    return 4.75 + ((c - HUMAN_BASELINE_HIGH) / (1.0 - HUMAN_BASELINE_HIGH)) * 0.25

def likelihood_to_accuracy_equivalent(likelihood_score):
    return float(80.0 + (likelihood_score / 5.0) * 20.0)


In [ ]:
# 7️⃣ Main scoring pass – per-record KSAC severity, salary/time cost, confidence & likelihood

if PRED_LABEL_COL not in merged_data.columns:
    raise ValueError(f"Predicted label column '{PRED_LABEL_COL}' not found in merged_data. "
                     "Please update PRED_LABEL_COL in the configuration cell.")

ksac_df  = resources.get("ksacs", pd.DataFrame())
salary_df = resources.get("salary_data", pd.DataFrame())
time_df   = resources.get("time_data", pd.DataFrame())

rows = []
salary_values = []
time_values = []

for idx, row in merged_data.iterrows():
    major_role = row.get(PRED_LABEL_COL, np.nan)
    if pd.isna(major_role):
        continue

    major_role_str = str(major_role).strip()

    error_severity, alternatives, best_neighbor_sim = calculate_error_severity(major_role_str, ksac_df)
    ksac_similarity = max(0.0, 1.0 - error_severity)

    salary_amount = calculate_salary_amount(major_role_str, salary_df)
    correction_hours = estimate_correction_time(error_severity, major_role_str, time_df)

    salary_values.append(salary_amount)
    time_values.append(correction_hours)

    tmp_row = row.copy()
    tmp_row["error_severity"] = error_severity
    tmp_row[PRED_LABEL_COL] = major_role_str
    confidence_score, confidence_reason = compute_confidence_components(tmp_row)

    rows.append({
        "row_index": idx,
        "major_role_group": major_role_str,
        "error_severity": error_severity,
        "ksac_best_neighbor_similarity": best_neighbor_sim,
        "salary_amount": salary_amount,
        "correction_time_hours": correction_hours,
        "confidence_score": confidence_score,
        "confidence_reason": confidence_reason,
        "alternatives": [r for r, s in alternatives],
        "alternative_similarities": [s for r, s in alternatives],
    })

intermediate_df = pd.DataFrame(rows)
print(f"✅ Computed intermediate metrics for {len(intermediate_df)} records.")

composite_scores = []
likelihood_scores = []
accuracy_equiv = []

for idx, rec in intermediate_df.iterrows():
    comp = compute_composite_quality(
        ksac_similarity=rec["ksac_best_neighbor_similarity"],
        salary_amount=rec["salary_amount"],
        correction_hours=rec["correction_time_hours"],
        confidence_score=rec["confidence_score"],
        all_salary_values=salary_values,
        all_hours_values=time_values,
    )
    composite_scores.append(comp)
    ls = map_composite_to_likelihood(comp)
    likelihood_scores.append(ls)
    accuracy_equiv.append(likelihood_to_accuracy_equivalent(ls))

intermediate_df["composite_quality"] = composite_scores
intermediate_df["likelihood_score"] = likelihood_scores
intermediate_df["accuracy_equivalent_pct"] = accuracy_equiv

results_df = merged_data.merge(
    intermediate_df.drop(columns=["major_role_group"]),
    left_index=True,
    right_on="row_index",
    how="left"
)

print("✅ Assembled results_df with original data plus likelihood metrics.")
print(results_df.head())

# Add run metadata columns for traceability
results_df["notebook_version"] = "v8.2-integrated"
results_df["run_tag"] = RUN_TAG
results_df["human_baseline_low"] = HUMAN_BASELINE_LOW
results_df["human_baseline_high"] = HUMAN_BASELINE_HIGH
results_df["weight_ksac_similarity"] = WEIGHT_KSAC_SIMILARITY
results_df["weight_salary_impact"] = WEIGHT_SALARY_IMPACT
results_df["weight_time_impact"] = WEIGHT_TIME_IMPACT
results_df["weight_confidence"] = WEIGHT_CONFIDENCE

main_results_path = os.path.join(RESULTS_PATH, "MNPS_Likelihood_Results.csv")
results_df.to_csv(main_results_path, index=False)
print(f"\n💾 Saved main likelihood results to: {main_results_path}")




In [ ]:
# 8️⃣ Batch summary & narrative insights

if "likelihood_score" not in results_df.columns:
    raise ValueError("likelihood_score not found in results_df. Ensure the main scoring cell ran successfully.")

summary = {}

summary["total_records"] = int(results_df["likelihood_score"].notna().sum())
summary["mean_likelihood"] = float(results_df["likelihood_score"].mean())
summary["median_likelihood"] = float(results_df["likelihood_score"].median())
summary["min_likelihood"] = float(results_df["likelihood_score"].min())
summary["max_likelihood"] = float(results_df["likelihood_score"].max())
summary["mean_accuracy_equiv_pct"] = float(results_df["accuracy_equivalent_pct"].mean())

def likelihood_band(ls):
    if pd.isna(ls):
        return "Unknown"
    if ls >= 4.5:
        return "Superhuman / Excellent"
    if ls >= 4.0:
        return "Human-level range"
    if ls >= 3.0:
        return "Slightly below human baseline"
    if ls >= 2.0:
        return "Below human baseline"
    return "Poor / Critical"

results_df["likelihood_band"] = results_df["likelihood_score"].apply(likelihood_band)
band_counts = results_df["likelihood_band"].value_counts(dropna=False).to_dict()
summary["band_counts"] = band_counts

total_salary_exposure = float(results_df["salary_amount"].sum(skipna=True))
total_correction_hours = float(results_df["correction_time_hours"].sum(skipna=True))

summary["total_salary_exposure"] = total_salary_exposure
summary["total_correction_hours"] = total_correction_hours

print("🔎 Batch Summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

lines = []
lines.append("MNPS Job Classification Likelihood & Severity – Batch Summary\n")
lines.append(f"Run Tag: {RUN_TAG}\n")
lines.append(f"Total Records Scored: {summary['total_records']}\n")
lines.append(f"Mean Likelihood (0–5): {summary['mean_likelihood']:.2f}\n")
lines.append(f"Median Likelihood (0–5): {summary['median_likelihood']:.2f}\n")
lines.append(f"Mean Accuracy-Equivalent (%): {summary['mean_accuracy_equiv_pct']:.1f}%\n")
lines.append("\nPerformance Bands:\n")
for band, count in band_counts.items():
    lines.append(f"  - {band}: {count} records\n")

lines.append("\nCost & Effort Signals (Approximate):\n")
lines.append(f"  - Total Salary Exposure of Classified Roles: ${summary['total_salary_exposure']:,.0f} per year\n")
lines.append(f"  - Estimated Total Hours to Review/Correct: {summary['total_correction_hours']:,.1f} hours\n")

summary_path = os.path.join(RESULTS_PATH, "MNPS_Likelihood_Batch_Summary.txt")
with open(summary_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print(f"\n💾 Saved narrative batch summary to: {summary_path}")


In [ ]:

# 8️⃣b Calibration audit – distribution of composite quality & likelihood

if "likelihood_score" not in results_df.columns or "composite_quality" not in results_df.columns:
    print("⚠️ Calibration audit skipped: composite_quality or likelihood_score missing.")
else:
    calib = {}
    ls = results_df["likelihood_score"].dropna()
    cq = results_df["composite_quality"].dropna()

    calib["n_records"] = int(len(ls))
    calib["likelihood_mean"] = float(ls.mean())
    calib["likelihood_std"] = float(ls.std())
    calib["composite_mean"] = float(cq.mean())
    calib["composite_std"] = float(cq.std())

    # Extremes: very low / very high likelihood
    calib["pct_likelihood_le_1_0"] = float((ls <= 1.0).mean() * 100.0)
    calib["pct_likelihood_ge_4_8"] = float((ls >= 4.8).mean() * 100.0)
    calib["pct_likelihood_human_band"] = float(((ls >= 4.0) & (ls <= 4.6)).mean() * 100.0)

    print("📏 Calibration audit (likelihood & composite quality):")
    for k, v in calib.items():
        print(f"  {k}: {v}")

    # Simple textual guidance
    guidance_lines = []
    if calib["pct_likelihood_ge_4_8"] > 20.0:
        guidance_lines.append(
            "- A large share of records have likelihood ≥ 4.8; consider tightening thresholds for 'auto-approve' or review."
        )
    if calib["pct_likelihood_le_1_0"] > 15.0:
        guidance_lines.append(
            "- Many records have likelihood ≤ 1.0; consider revisiting prompt, KSAC mapping, or model behavior for these tails."
        )
    if not guidance_lines:
        guidance_lines.append("- Calibration appears reasonable; no obvious extreme clustering in likelihood scores.")

    calib_report_path = os.path.join(RESULTS_PATH, "MNPS_Likelihood_Calibration_Audit.txt")
    with open(calib_report_path, "w", encoding="utf-8") as f:
        f.write("MNPS Likelihood Calibration Audit\n\n")
        for k, v in calib.items():
            f.write(f"{k}: {v}\n")
        f.write("\nGuidance:\n")
        for line in guidance_lines:
            f.write(f"{line}\n")

    print(f"\n💾 Saved calibration audit to: {calib_report_path}")



In [ ]:
# 9️⃣ Severity-Weighted Evaluation & Confusion Analysis (integrated v2 logic)

import numpy as np

print("🔍 Running severity-weighted evaluation based on current results_df...\n")

eval_df = results_df.copy()

if TRUE_LABEL_COL not in eval_df.columns:
    raise ValueError(
        f"True label column '{TRUE_LABEL_COL}' not found in results_df. "
        "Add this column to your Job_Classifications_Batch.csv or merged_data, "
        "or update TRUE_LABEL_COL in the configuration cell."
    )

if PRED_LABEL_COL not in eval_df.columns:
    raise ValueError(
        f"Pred label column '{PRED_LABEL_COL}' not found in results_df. "
        "Update PRED_LABEL_COL in the configuration cell to match the predicted major role group column."
    )

def compute_true_pred_similarity(row):
    true_label = row.get(TRUE_LABEL_COL, None)
    pred_label = row.get(PRED_LABEL_COL, None)
    if pd.isna(true_label) or pd.isna(pred_label):
        return np.nan
    return calculate_ksac_similarity(str(true_label), str(pred_label), resources.get("ksacs", pd.DataFrame()))

eval_df["similarity_score"] = eval_df.apply(compute_true_pred_similarity, axis=1)

def normalize_column(df: pd.DataFrame, col_name: str):
    '''
    Return (col_name, max_val) with basic safety handling.
    '''
    if col_name is None or col_name not in df.columns:
        return None, 0.0
    max_val = float(df[col_name].max())
    if max_val <= 0 or np.isnan(max_val):
        return col_name, 0.0
    return col_name, max_val

salary_col = None
for c in eval_df.columns:
    cl = c.lower()
    if "salary_amount" in cl or (cl.startswith("salary") and "score" not in cl):
        salary_col = c
        break

hours_col = None
for c in eval_df.columns:
    cl = c.lower()
    if "correction_time_hours" in cl or "correction_hours" in cl or cl == "hours":
        hours_col = c
        break

salary_col, max_salary = normalize_column(eval_df, salary_col)
hours_col, max_hours   = normalize_column(eval_df, hours_col)

print(f"Using salary column: {salary_col} (max={max_salary})")
print(f"Using hours column:  {hours_col} (max={max_hours})")

def compute_severity_cost_index(row):
    sim = row.get("similarity_score", np.nan)
    if np.isnan(sim):
        sim = 0.0
    sim = min(max(sim, 0.0), 1.0)
    dissimilarity = 1.0 - sim

    if salary_col is not None and max_salary > 0:
        sal_raw = row.get(salary_col, 0.0)
        sal_norm = min(max(sal_raw / max_salary, 0.0), 1.0)
    else:
        sal_norm = 0.0

    if hours_col is not None and max_hours > 0:
        hrs_raw = row.get(hours_col, 0.0)
        hrs_norm = min(max(hrs_raw / max_hours, 0.0), 1.0)
    else:
        hrs_norm = 0.0

    severity_component = (
        SEVERITY_SALARY_WEIGHT * sal_norm +
        SEVERITY_TIME_WEIGHT   * hrs_norm
    )

    return float(dissimilarity * severity_component)

eval_df["severity_cost_index"] = eval_df.apply(compute_severity_cost_index, axis=1)

def categorize_severity_band(series: pd.Series):
    non_na = series.dropna()
    if len(non_na) == 0:
        return series.apply(lambda _: "Unknown")

    q50 = non_na.quantile(0.50)
    q75 = non_na.quantile(0.75)
    q90 = non_na.quantile(0.90)

    def _band(val):
        if pd.isna(val):
            return "Unknown"
        if val <= q50:
            return "Low"
        if val <= q75:
            return "Medium"
        if val <= q90:
            return "High"
        return "Extreme"

    return series.apply(_band)

eval_df["severity_band"] = categorize_severity_band(eval_df["severity_cost_index"])

print("✅ Severity index and bands computed.")
print(eval_df[[TRUE_LABEL_COL, PRED_LABEL_COL, "severity_cost_index", "severity_band"]].head())

confusion_counts = pd.crosstab(
    eval_df[TRUE_LABEL_COL],
    eval_df[PRED_LABEL_COL],
    dropna=False
)

confusion_severity = pd.crosstab(
    eval_df[TRUE_LABEL_COL],
    eval_df[PRED_LABEL_COL],
    values=eval_df["severity_cost_index"],
    aggfunc="sum",
    dropna=False
).fillna(0.0)

print("\nStandard Confusion Matrix (Counts):")
display(confusion_counts)

print("\nSeverity-Weighted Confusion Matrix (Sum of severity_cost_index):")
display(confusion_severity)

is_correct = eval_df[TRUE_LABEL_COL] == eval_df[PRED_LABEL_COL]
is_error   = ~is_correct

total_records = len(eval_df)
total_errors  = int(is_error.sum())
total_severity_error = float(eval_df.loc[is_error, "severity_cost_index"].sum())
avg_severity_per_error = float(eval_df.loc[is_error, "severity_cost_index"].mean()) if total_errors > 0 else 0.0

print("\nSeverity-Weighted Misclassification Metrics:")
print(f"  Total records:                      {total_records}")
print(f"  Total errors:                       {total_errors}")
print(f"  Total severity-weighted error cost: {total_severity_error:.3f}")
print(f"  Average severity cost per error:    {avg_severity_per_error:.3f}")

error_df = eval_df[is_error].copy()
severity_by_true = (
    error_df.groupby(TRUE_LABEL_COL)["severity_cost_index"]
    .agg(["count", "sum", "mean"])
    .rename(columns={"count": "error_count",
                     "sum": "total_severity_cost",
                     "mean": "avg_severity_per_error"})
)


print("\nSeverity-weighted error metrics by TRUE class:")
display(severity_by_true)

# KSAC group analysis (within-group vs cross-group errors) using MNPS_Role_Groups_by_KSAC_Similarity_FINAL
role_groups_df = resources.get("role_groups", pd.DataFrame())

def build_role_group_lookup(df):
    if df is None or df.empty:
        return {}
    tmp = df.copy()
    tmp.columns = tmp.columns.str.strip()
    role_cols = [c for c in tmp.columns if "role" in c.lower()]
    group_cols = [c for c in tmp.columns if "group" in c.lower()]
    if not role_cols or not group_cols:
        return {}
    rcol = role_cols[0]
    gcol = group_cols[0]
    mapping = {}
    for _, r in tmp[[rcol, gcol]].dropna().iterrows():
        mapping[str(r[rcol]).strip()] = str(r[gcol]).strip()
    return mapping

role_group_lookup = build_role_group_lookup(role_groups_df)

def get_group(label):
    if not role_group_lookup:
        return None
    return role_group_lookup.get(str(label).strip())

# Annotate error_df with true/pred groups and cross-group flag
error_df["true_group"] = error_df[TRUE_LABEL_COL].apply(get_group)
error_df["pred_group"] = error_df[PRED_LABEL_COL].apply(get_group)
error_df["cross_group_error"] = (
    (error_df["true_group"].notna()) &
    (error_df["pred_group"].notna()) &
    (error_df["true_group"] != error_df["pred_group"])
)

# Aggregate severity by within vs cross-group
severity_by_group_relation = (
    error_df.groupby("cross_group_error")["severity_cost_index"]
    .agg(["count", "sum", "mean"])
    .rename(index={False: "within_group_or_unknown", True: "cross_group"})
)

print("\nSeverity-weighted error metrics by KSAC group relation (within vs cross-group):")
display(severity_by_group_relation)

confusion_severity_mean = pd.crosstab(
    error_df[TRUE_LABEL_COL],
    error_df[PRED_LABEL_COL],
    values=error_df["severity_cost_index"],
    aggfunc="mean",
    dropna=False
).fillna(0.0)

confusion_severity_mean = pd.crosstab(
    error_df[TRUE_LABEL_COL],
    error_df[PRED_LABEL_COL],
    values=error_df["severity_cost_index"],
    aggfunc="mean",
    dropna=False
).fillna(0.0)

print("\nMean severity_cost_index for each misclassification pair (true → predicted):")
display(confusion_severity_mean)

# Attach run metadata to evaluation frame
eval_df["notebook_version"] = "v8.2-integrated"
eval_df["run_tag"] = RUN_TAG
eval_df["human_baseline_low"] = HUMAN_BASELINE_LOW
eval_df["human_baseline_high"] = HUMAN_BASELINE_HIGH

eval_out_path = os.path.join(RESULTS_PATH, "MNPS_Eval_with_Severity_Index.csv")
eval_df.to_csv(eval_out_path, index=False)
print(f"\n💾 Saved evaluation file with severity index to: {eval_out_path}")

counts_path = os.path.join(RESULTS_PATH, "MNPS_Confusion_Counts.csv")
confusion_counts.to_csv(counts_path)
print(f"💾 Saved confusion counts matrix to: {counts_path}")

severity_path = os.path.join(RESULTS_PATH, "MNPS_Confusion_SeverityWeighted.csv")
confusion_severity.to_csv(severity_path)
print(f"💾 Saved severity-weighted confusion matrix to: {severity_path}")





In [ ]:

# 🔟a Top N records to review – severity- and KSAC-aware queue

# You can adjust this number to change how many records appear in the review list.
top_n = 50

# Prefer the evaluation frame (which includes severity_cost_index, severity_band, cross_group_error)
try:
    df = eval_df.copy()
except NameError:
    df = None

if df is None or "severity_cost_index" not in (df.columns if df is not None else []):
    print("⚠️ Top N review list skipped: eval_df with severity_cost_index not found. "
          "Please run the severity-weighted evaluation cell first.")
else:
    # Focus on misclassifications, if true/pred columns are available
    if TRUE_LABEL_COL in df.columns and PRED_LABEL_COL in df.columns:
        error_mask = df[TRUE_LABEL_COL] != df[PRED_LABEL_COL]
        df_errors = df[error_mask].copy()
    else:
        df_errors = df.copy()

    # Ensure we have the key ranking columns
    if "likelihood_score" not in df_errors.columns:
        df_errors["likelihood_score"] = np.nan
    if "cross_group_error" not in df_errors.columns:
        df_errors["cross_group_error"] = False

    # Rank: cross-group errors first, then highest severity, then lowest likelihood
    df_errors["cross_group_error"] = df_errors["cross_group_error"].fillna(False)

    df_ranked = df_errors.sort_values(
        by=["cross_group_error", "severity_cost_index", "likelihood_score"],
        ascending=[False, False, True]
    )

    top_review = df_ranked.head(top_n).copy()

    # Select a useful subset of columns for the review queue
    cols = []
    for c in [
        TRUE_LABEL_COL,
        PRED_LABEL_COL,
        "severity_cost_index",
        "severity_band",
        "likelihood_score",
        "accuracy_equivalent_pct",
        "error_severity",
        "confidence_score",
        "cross_group_error",
        "similarity_score",
        "salary_amount",
        "correction_time_hours",
        "row_index"
    ]:
        if c in top_review.columns and c not in cols:
            cols.append(c)

    review_view = top_review[cols] if cols else top_review

    print(f"📋 Top {min(top_n, len(top_review))} records to review "
          f"(prioritized by cross_group_error, severity_cost_index, low likelihood):")
    display(review_view)

    # Save to results folder
    review_path = os.path.join(RESULTS_PATH, f"MNPS_Top_Review_List_Top{top_n}.csv")
    review_view.to_csv(review_path, index=False)
    print(f"💾 Saved prioritized review list to: {review_path}")



In [ ]:

# 🔟 Visual diagnostics – likelihood, severity, and cost patterns

import matplotlib.pyplot as plt

# 1) Histogram of likelihood scores
if "likelihood_score" in results_df.columns:
    plt.figure()
    results_df["likelihood_score"].dropna().plot(kind="hist", bins=20)
    plt.title("Distribution of Likelihood Scores (0–5)")
    plt.xlabel("Likelihood score")
    plt.ylabel("Count")
    plt.show()
else:
    print("⚠️ Skipping likelihood histogram: likelihood_score not found in results_df.")

# 2) Bar chart of total severity_cost_index by TRUE major group
if "severity_cost_index" in results_df.columns and TRUE_LABEL_COL in results_df.columns:
    severity_by_true_chart = (
        results_df.groupby(TRUE_LABEL_COL)["severity_cost_index"]
        .sum()
        .sort_values(ascending=False)
    )
    plt.figure()
    severity_by_true_chart.plot(kind="bar")
    plt.title("Total Severity Cost Index by TRUE Major Role Group")
    plt.xlabel("True Major Role Group")
    plt.ylabel("Total severity_cost_index")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Skipping severity-by-true chart: severity_cost_index or TRUE_LABEL_COL missing.")

# 3) Heatmap-style image of severity-weighted confusion matrix (if available)
try:
    severity_matrix_path = os.path.join(RESULTS_PATH, "MNPS_Confusion_SeverityWeighted.csv")
    if os.path.exists(severity_matrix_path):
        conf_sev = pd.read_csv(severity_matrix_path, index_col=0)
        plt.figure()
        plt.imshow(conf_sev.values)
        plt.title("Severity-Weighted Confusion Matrix")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.xticks(range(len(conf_sev.columns)), conf_sev.columns, rotation=45, ha="right")
        plt.yticks(range(len(conf_sev.index)), conf_sev.index)
        plt.tight_layout()
        plt.show()
    else:
        print("ℹ️ Severity-weighted confusion CSV not found; run the evaluation cell first to generate it.")
except Exception as e:
    print("⚠️ Could not render severity-weighted confusion heatmap:", e)

